# Etap EDA - krok 3: jakość kluczowych pól

Cel: ocenić jakość pól kluczowych dla dalszej analizy. Przede wszystkim `stop_sequence`
(warunek konieczny dla liczenia przyrostu opóźnienia na odcinku) oraz samych wartości
opóźnienia (`arrival_delay`, `departure_delay`).

In [1]:
import duckdb
import zipfile
import tempfile
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))
from config import RAW, LOGS, STATIC

print("RAW   :", RAW)
print("STATIC:", STATIC)

RAW   : C:\kmk-analiza\data\raw
STATIC: C:\kmk-analiza\data\static


## `stop_sequence` - skala braków

Wiemy już, że to pole często jest puste. Zmierzymy dokładnie, jak bardzo - ogólnie i per feed.

In [2]:
zapytanie = f"""
SELECT
    feed,
    count(*) AS wszystkie,
    count(stop_sequence) AS ma_stop_sequence,
    round(100.0 * count(stop_sequence) / count(*), 2) AS proc_stop_seq,
    round(100.0 * count(direction_id) / count(*), 2) AS proc_direction,
    round(100.0 * count(stop_schedule_relationship) / count(*), 2) AS proc_stop_sched
FROM read_parquet('{RAW.as_posix()}/trip_updates/**/*.parquet',
                  hive_partitioning = true, union_by_name = true)
GROUP BY feed
ORDER BY feed
"""
print(duckdb.query(zapytanie).to_df().to_string(index=False))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

feed  wszystkie  ma_stop_sequence  proc_stop_seq  proc_direction  proc_stop_sched
   A  110274916          21858210          19.82           10.27            10.27
   M   44596162          44596162         100.00          100.00           100.00
   T   68745627                 0           0.00            0.00           100.00


**Wynik**: feed M ma 100% wypełnienia.
Feed T ma 0% `stop_sequence`, ale 100% `stop_schedule_relationship`. **Feed A tylko ~19,8%.** To realne, poważne ograniczenie głównego strumienia danych.

## Czy `stop_sequence` da się odtworzyć z rozkładu statycznego?

`stop_times.txt` w paczkach GTFS zawiera `trip_id`, `stop_id` i `stop_sequence`. Pełną,
oficjalną kolejność przystanków. Testujemy, czy `trip_id` z danych RT pasują do `trip_id`
w rozkładzie.

In [3]:
paczki_A = sorted(STATIC.glob("GTFS_KRK_A_*.zip"))
print("Znalezione paczki A:", len(paczki_A))
for p in paczki_A:
    print("  ", p.name)

Znalezione paczki A: 9
   GTFS_KRK_A_01a62bb9a835.zip
   GTFS_KRK_A_1dda9ab3ee73.zip
   GTFS_KRK_A_23f8ee7e8fd7.zip
   GTFS_KRK_A_68319bc265ab.zip
   GTFS_KRK_A_8bfcf5c34b72.zip
   GTFS_KRK_A_986071d33634.zip
   GTFS_KRK_A_9e9a62ed9826.zip
   GTFS_KRK_A_a20ddf52be5c.zip
   GTFS_KRK_A_ad37a5cfca38.zip


## Dobór właściwej wersji rozkładu

Nazwy plików `GTFS_KRK_*.zip` zawierają losowy hash. Właściwe źródło daty, która da chronologię, to `feed_info.txt` wewnątrz paczki (`feed_start_date`, `feed_end_date`, `feed_version`).

In [4]:
for p in paczki_A:
    with zipfile.ZipFile(p) as z:
        with z.open("feed_info.txt") as f:
            tresc = f.read().decode("utf-8")
    print(f"--- {p.name} ---")
    print(tresc)

--- GTFS_KRK_A_01a62bb9a835.zip ---
feed_publisher_name,feed_publisher_url,feed_lang,feed_start_date,feed_end_date,feed_version,feed_contact_email,feed_contact_url
"Miejskie Przedsiębiorstwo Komunikacyjne S.A. w Krakowie",https://gtfs.ztp.krakow.pl,pl,20261013,20270131,"20261013_20261017","webmaster@mpk.krakow.pl",""

--- GTFS_KRK_A_1dda9ab3ee73.zip ---
feed_publisher_name,feed_publisher_url,feed_lang,feed_start_date,feed_end_date,feed_version,feed_contact_email,feed_contact_url
"Miejskie Przedsiębiorstwo Komunikacyjne S.A. w Krakowie",https://gtfs.ztp.krakow.pl,pl,20261010,20270131,"20261010_20261013","webmaster@mpk.krakow.pl",""

--- GTFS_KRK_A_23f8ee7e8fd7.zip ---
feed_publisher_name,feed_publisher_url,feed_lang,feed_start_date,feed_end_date,feed_version,feed_contact_email,feed_contact_url
"Miejskie Przedsiębiorstwo Komunikacyjne S.A. w Krakowie",https://gtfs.ztp.krakow.pl,pl,20261005,20270131,"20261005_20261010","webmaster@mpk.krakow.pl",""

--- GTFS_KRK_A_68319bc265ab.zip ---
feed

Na 23 października przypada wersja `GTFS_KRK_A_a20ddf52be5c.zip`.
Testujemy dopasowanie `trip_id` właśnie na tej parze: paczka i dzień 23 października.

In [5]:
sciezka_zip = next(p for p in paczki_A if p.name == "GTFS_KRK_A_a20ddf52be5c.zip")
print("Uzywam paczki:", sciezka_zip.name)

folder_tmp = Path(tempfile.mkdtemp())
with zipfile.ZipFile(sciezka_zip) as z:
    z.extract("stop_times.txt", path=folder_tmp)
sciezka_stop_times = folder_tmp / "stop_times.txt"

zapytanie = f"""
WITH rozklad AS (
    SELECT DISTINCT trip_id
    FROM read_csv_auto('{sciezka_stop_times.as_posix()}')
),
rt AS (
    SELECT DISTINCT trip_id
    FROM read_parquet('{RAW.as_posix()}/trip_updates/**/*.parquet',
                      hive_partitioning = true, union_by_name = true)
    WHERE feed = 'A' AND date = '2026-10-23'
)
SELECT
    (SELECT count(*) FROM rt) AS unikalne_trip_id_rt,
    (SELECT count(*) FROM rt WHERE trip_id IN (SELECT trip_id FROM rozklad)) AS dopasowane,
    round(100.0 * (SELECT count(*) FROM rt WHERE trip_id IN (SELECT trip_id FROM rozklad))
          / (SELECT count(*) FROM rt), 2) AS proc_dopasowania
"""
print(duckdb.query(zapytanie).to_df().to_string(index=False))

Uzywam paczki: GTFS_KRK_A_a20ddf52be5c.zip
 unikalne_trip_id_rt  dopasowane  proc_dopasowania
               10184       10125             99.42


**Wynik: 99,42% dopasowania.** Potwierdza to, że `stop_sequence` da się odtworzyć łącząc RT
z rozkładem statycznym po `trip_id` i `stop_id`, pod warunkiem wzięcia wersji rozkładu
obowiązującej w dniu obserwacji. `trip_id` koduje `service_date`, które po północy bywa
inne niż partycja `date`. Będzie to istotne przy łączeniu blisko granic dni.

## Jakość pola opóźnienia (`arrival_delay`, `departure_delay`)

Sprawdzamy: wypełnienie, zakres wartości, medianę (rozkład jest prawoskośny) i czy wartości są mimo typu `DOUBLE` faktycznie całkowite.

In [6]:
zapytanie = f"""
SELECT
    feed,
    count(*) AS wszystkie,
    count(arrival_delay) AS ma_arrival_delay,
    round(100.0 * count(arrival_delay) / count(*), 2) AS proc_wypelnienia,
    min(arrival_delay) AS min_opoznienie,
    max(arrival_delay) AS max_opoznienie,
    median(arrival_delay) AS mediana,
    sum(CASE WHEN arrival_delay <> floor(arrival_delay) THEN 1 ELSE 0 END) AS niecalkowite
FROM read_parquet('{RAW.as_posix()}/trip_updates/**/*.parquet',
                  hive_partitioning = true, union_by_name = true)
GROUP BY feed
ORDER BY feed
"""
print(duckdb.query(zapytanie).to_df().to_string(index=False))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

feed  wszystkie  ma_arrival_delay  proc_wypelnienia  min_opoznienie  max_opoznienie  mediana  niecalkowite
   A  110274916         105903659             96.04         -1362.0         21261.0      0.0           0.0
   M   44596162          43082922             96.61          -308.0          3194.0      0.0           0.0
   T   68745627          66025616             96.04             0.0             0.0      0.0           0.0


In [7]:
zapytanie = f"""
SELECT
    feed,
    count(DISTINCT arrival_delay) AS unikalne_wartosci,
    approx_quantile(arrival_delay, 0.95) AS p95,
    approx_quantile(arrival_delay, 0.99) AS p99,
    approx_quantile(arrival_delay, 0.999) AS p999,
    sum(CASE WHEN abs(arrival_delay) > 3600 THEN 1 ELSE 0 END) AS powyzej_1h,
    sum(CASE WHEN abs(arrival_delay) > 7200 THEN 1 ELSE 0 END) AS powyzej_2h
FROM read_parquet('{RAW.as_posix()}/trip_updates/**/*.parquet',
                  hive_partitioning = true, union_by_name = true)
GROUP BY feed
ORDER BY feed
"""
print(duckdb.query(zapytanie).to_df().to_string(index=False))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

feed  unikalne_wartosci        p95        p99        p999  powyzej_1h  powyzej_2h
   A               4350   0.000000 180.224813 2113.023971     47983.0     39049.0
   M               2776 357.085336 847.019073 1578.111706         0.0         0.0
   T                  1   0.000000   0.000000    0.000000         0.0         0.0


In [8]:
zapytanie = f"""
SELECT
    count(*) AS rekordy_powyzej_2h,
    count(DISTINCT trip_id) AS unikalne_kursy,
    count(DISTINCT route_id) AS unikalne_linie,
    min(arrival_delay) AS min_w_grupie,
    max(arrival_delay) AS max_w_grupie
FROM read_parquet('{RAW.as_posix()}/trip_updates/**/*.parquet',
                  hive_partitioning = true, union_by_name = true)
WHERE feed = 'A' AND arrival_delay > 7200
"""
print(duckdb.query(zapytanie).to_df().to_string(index=False))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

 rekordy_powyzej_2h  unikalne_kursy  unikalne_linie  min_w_grupie  max_w_grupie
              39049              25               3        7204.0       21261.0


In [9]:
zapytanie = f"""
SELECT
    trip_id, route_id,
    count(*) AS ile_obserwacji,
    min(arrival_delay) AS min_opoznienie,
    max(arrival_delay) AS max_opoznienie,
    min(date) AS pierwszy_dzien,
    max(date) AS ostatni_dzien
FROM read_parquet('{RAW.as_posix()}/trip_updates/**/*.parquet',
                  hive_partitioning = true, union_by_name = true)
WHERE feed = 'A' AND arrival_delay > 7200
GROUP BY trip_id, route_id
ORDER BY ile_obserwacji DESC
LIMIT 15
"""
print(duckdb.query(zapytanie).to_df().to_string(index=False))

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

                trip_id route_id  ile_obserwacji  min_opoznienie  max_opoznienie pierwszy_dzien ostatni_dzien
20261024_1_129660792_19      146            2799         19748.0         21173.0     2026-10-26    2026-10-26
20261024_1_129660791_18      146            2716         19715.0         21090.0     2026-10-26    2026-10-26
20261024_1_129660793_20      146            2458         19818.0         21137.0     2026-10-26    2026-10-26
20261024_1_129660790_17      146            2123         19918.0         20990.0     2026-10-26    2026-10-26
20261024_1_129660785_12      146            2053         19654.0         21002.0     2026-10-26    2026-10-26
20261024_1_129660787_14      146            1954         19712.0         21089.0     2026-10-26    2026-10-26
20261024_1_129660786_13      146            1953         19686.0         21134.0     2026-10-26    2026-10-26
 20261024_1_129660781_8      146            1911         19873.0         21121.0     2026-10-26    2026-10-26
20261024_1

## Wnioski

1. **`stop_sequence` brakuje głównie w feedzie A** ~19,8%, feed M 100%, feed T 0%.
2. **Problem jest do rozwiązania**: Odtworzenie przez złączenie z rozkładem statycznym po
   właściwej wersji (`feed_info.txt`), potwierdzone na próbce: **99,42% dopasowania**.
3. **Jakość opóźnień (`arrival_delay`)**:
   - Typ `DOUBLE`, ale wartości w 100% całkowite. Jest to prawdopodobnie artefakt zapisu, który jest bez znaczenia.
   - Wypełnienie ~96% we wszystkich feedach (reszta to prognozy dla przyszłych przystanków, jeszcze
     bez wartości).
   - **Feed A, rdzeń rozkładu do P99 (3 min) jest poprawny**, ale ma długi ogon do max 21261s (5.9h).
   - **Zidentyfikowano i zrozumiano źródło ekstremów:** 39049 rekordów (z 110 mln) pochodzi od
     zaledwie **25 unikalnych kursów** linii **146** z dnia **26.10**, prawdopodobnie odwołanych lub
     mocno zakłóconych i niepoprawnie oznaczonych jako aktywne (pole `trip_schedule_relationship`,
     które powinno to rozróżniać, jest w feedzie A puste). Wartości rosną płynnie w czasie, więc to nie
     błąd zapisu, tylko realne zachowanie źródła przy kursach wypadających z normalnego cyklu.
   - **Decyzja do preprocessingu:** przyjąć próg odcięcia jako wartość nierealną dla pojedynczego
     kursu i wycinać lub oznaczać osobno. To jeszcze do potwierdzenia na kolejnym etapie.